# 2b Call Different Models and Routed Resources

**Purpose:**
Use one gateway, one token, and one request format to reach several models, then let a **Routed Resource** pick the model for you.

**What we're doing**

We send the same prompt to every model in your profile and compare the answers. Then we call a Routed Resource, a single model name that the gateway sends to different models based on rules, and watch it choose.

**Why we're doing it**

* In `02_Use_the_LLM_Gateway.ipynb` you called one model through the gateway. Real applications use several: a small, cheap model for easy requests and a larger one for hard requests.
* Without the gateway, each model would mean another URL, another credential, and another code path. With it, switching models is a one-word change.
* A Routed Resource moves the choice of model out of client code and into a rule your administrator controls

**What to look for**

* The only thing that changes between models is the `"model"` value
* Different models answering the same prompt with different length, quality, and latency
* One alias, two different models answering, depending on the request

**Output**

A side-by-side comparison of the models in your profile, and a Routed Resource sending requests to different models.


## 2.8 How the Gateway Finds a Model

The LLM Gateway has three building blocks. You met the profile in `02_Use_the_LLM_Gateway.ipynb`; here are all three:

| Object | What it is | Example |
|---|---|---|
| **Resource** | A connection to one upstream model: a provider, a model ID, an endpoint URL, and an API key | Your Section 1 vLLM endpoint, or an external provider's model |
| **Routed Resource** | A rule set that sends each request to one of several Resources | `smart-router` |
| **Profile** | The models a set of users may call. Each entry points to a Resource or a Routed Resource and gives it an **alias name**. | `frank-test` |

Clients see only the profile's aliases. They never see a Resource's upstream URL or API key:

```
                 X-ClearML-Profile: frank-test
client ── "model": "<alias>" ──▶ LLM Gateway ──▶ Profile ──▶ Resource ─────────────▶ one upstream model
                                                     │
                                                     └────▶ Routed Resource ──rules──▶ Resource A or Resource B
```

> **Note:** Every Resource has its own **Service provider** setting, which tells the gateway how to talk to that upstream model. That is why the client sends the same OpenAI-compatible request to every alias, whatever the provider is.

## 2.9 Load the Token and the Gateway URL

This is the same setup as `02_Use_the_LLM_Gateway.ipynb`: the token and the gateway URL come from `config.py`, and the profile name is set in the cell.

In [ ]:
import sys
import time
sys.path.insert(0, "..")
from config import KLIQUE_GATEWAY_TOKEN as token, KLIQUE_LLM_GATEWAY_URL as gateway_url
import requests

PROFILE = "frank-test"  # Your LLM Gateway profile

print(f"LLM Gateway: {gateway_url}")
print(f"Profile:     {PROFILE}")
print(f"Token:       {token[:8]}...")

The helper below is the `chat()` function from `02_Use_the_LLM_Gateway.ipynb`, with one addition: it also returns how long the call took, so you can compare models.

In [ ]:
def chat(prompt, model_name, max_tokens=256, profile_name=PROFILE, timeout=120):
    """Send a chat completion through the LLM Gateway. Return (parsed reply, seconds taken)."""
    start = time.perf_counter()
    r = requests.post(
        f"{gateway_url}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {token}",
            "X-ClearML-Profile": profile_name,
        },
        json={
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        },
        timeout=timeout,
    )
    elapsed = time.perf_counter() - start
    if not r.ok:
        # Show the gateway's explanation, e.g. 404 model_not_found.
        print(f"❌ HTTP {r.status_code}: {r.text[:300]}")
    r.raise_for_status()
    return r.json(), elapsed

## 2.10 List the Models in Your Profile

The gateway does not serve `/v1/models`, so get the list from the WebApp. Open your user **Settings**, go to your **Profile** tab, and find **Your AI Endpoints**. It lists every alias your profile gives you.

<!-- screenshot: images/your_ai_endpoints.png (Profile tab, Your AI Endpoints listing the profile's aliases) -->

Copy the aliases into the cell below, with a short note on what each one is. Replace the examples with your own. Keep your Section 1 model in the list.

In [ ]:
MODELS = {
    "vllm-qwen-franktest": "Section 1: Qwen2.5-1.5B on vLLM, on Klique",
    "large-model":         "A larger model in your profile",      # replace with a real alias
    "external-model":      "An external provider's model",        # replace with a real alias, or delete
}

## 2.11 Same Prompt, Different Models

Now send one prompt to every alias. Look at the request inside `chat()`: the URL, the headers, and the body are identical for every model, except for `"model"`.

In [ ]:
prompt = "Explain in two sentences why a team would deploy its own model instead of only using a hosted API."

results = []
for alias, description in MODELS.items():
    print(f"🤖 {alias}  ({description})")
    try:
        reply, seconds = chat(prompt, alias)
    except requests.HTTPError:
        print()
        continue
    answer = reply["choices"][0]["message"]["content"].strip()
    tokens = reply["usage"]["completion_tokens"]
    results.append((alias, reply["model"], seconds, tokens))
    print(f"✅ {answer}\n")

print(f"{'Alias':<24} {'Served by':<32} {'Seconds':>8} {'Output tokens':>14}")
for alias, served_by, seconds, tokens in results:
    print(f"{alias:<24} {served_by:<32} {seconds:>8.1f} {tokens:>14}")

Read the table at the bottom:

- **Served by** is the model that actually answered. For your Section 1 alias it is the Model Endpoint Name, as you saw in `02_Use_the_LLM_Gateway.ipynb`. For other aliases it is the model ID the Resource sends upstream.
- **Seconds** and **Output tokens** show the trade-off. A small model on your own GPU is usually fast and cheap, but a larger model may give a more careful answer to the same prompt.

> **Checkpoint:** at least two models answered, and their *Served by* values differ. Any alias that printed `❌ HTTP 404` is not in your profile under that name; fix it in 2.10.

Your application did not need a new URL, a new key, or new code for any of these models. That is what the profile buys you. But the application still had to **pick** the alias. The rest of this notebook moves that choice into the gateway.


## 2.12 What a Routed Resource Does

A **Routed Resource** is a single alias that stands for several Resources. When a request arrives, the gateway checks the Routed Resource's rules and forwards the request to the Resource the winning rule names.

Each rule has five fields:

| Field | What it does | Example |
|---|---|---|
| **Priority** | Ranks the rule. Rules are checked from highest priority to lowest, and the first match wins. | `10` |
| **Route on** | The request attribute to test, such as the requested model or the maximum output tokens | max output tokens |
| **Operator** | How to compare the attribute with the value | greater than |
| **Value** | What to compare against | `512` |
| **Route to resource** | Where matching requests go | the larger model's Resource |

In this lab, the Routed Resource is called **`smart-router`** and uses one idea: *a request that asks for a long answer is a harder request*. Long answers go to the larger model, and everything else goes to your Section 1 vLLM model.

| Priority | Route on | Operator | Value | Route to resource |
|---|---|---|---|---|
| 10 | max output tokens | greater than | `512` | The larger model's Resource |
| 1 | max output tokens | less than or equal to | `512` | Your Section 1 vLLM Resource |

The two rules together cover every value of `max_tokens`, so every request matches one of them.

> **Note:** The rules read the request, not the prompt's meaning. A router with these rules does not know whether a question is hard; it knows only what the request asks for. Choose attributes your clients set on purpose.

## 2.13 Set Up the Routed Resource

Routed Resources are created in the LLM Gateway settings, usually by your Klique administrator. If `smart-router` is not already in your profile, give your administrator the two rules from 2.12 and ask them to:

1. Open **Settings → LLM Gateway** and click **Add Routed Resource**.
2. Set **Name** to `smart-router`, and add the two rules from 2.12.
3. Open your profile and click **+ Add Model**. Set **Target type** to **Routed Resource**, choose `smart-router` as the **Target**, and keep `smart-router` as the **Alias name**.

<!-- screenshot: images/add_routed_resource.png (Add Routed Resource dialog with the two rules) -->

<!-- screenshot: images/profile_routed_resource.png (profile's model list with smart-router as a Routed Resource target) -->

Then set the alias in the cell below. If your administrator chose a different alias, use theirs.

In [ ]:
ROUTER = "smart-router"  # The Routed Resource's alias in your profile

## 2.14 Watch the Router Choose

Send the same prompt to the same alias twice. The only difference is `max_tokens`: 256, then 1024. Watch **Served by**.

In [ ]:
prompt = "List three things to check before putting a model endpoint into production."

for max_tokens in (256, 1024):
    reply, seconds = chat(prompt, ROUTER, max_tokens=max_tokens)
    print(f"max_tokens={max_tokens:<5}  →  served by {reply['model']}  ({seconds:.1f}s)")

The client asked for `smart-router` both times. With `max_tokens=256`, the priority-1 rule matched and your Section 1 model answered. With `max_tokens=1024`, the priority-10 rule matched first and the larger model answered.

> **Checkpoint:** the two lines name two different models. If both name the same model, check the rules' **Value** and **Operator** with your administrator.

Now read the answers from both models:

In [ ]:
for max_tokens in (256, 1024):
    reply, _ = chat(prompt, ROUTER, max_tokens=max_tokens)
    print(f"--- max_tokens={max_tokens}, served by {reply['model']} ---")
    print(reply["choices"][0]["message"]["content"].strip(), "\n")

## 2.15 Why This Matters

Look at what the client code knows: one URL, one token, one profile, and one model name, `smart-router`. It does not know which models exist behind that name or what the rules are.

That means the routing can change without touching any client:

- **Swap a model.** Point a rule at a new Resource, and every client gets the new model on its next request.
- **Change the threshold.** Raise `512` to `1024`, and more traffic stays on your own GPU.
- **Add a tier.** Add a rule with a higher priority for a third model.

Every one of these is a change in the LLM Gateway, made once, and all of it is metered: open the **AI Usage** dashboard to see the calls from this notebook, split by model.

<!-- screenshot: images/ai_usage_by_model.png (AI Usage dashboard, usage broken down by model) -->


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `404` with `model_not_found` for one alias | That alias is not in your profile, or it is spelled differently. Compare it with **Your AI Endpoints**. |
| `404` for `smart-router` | The Routed Resource exists but has not been added to your profile, or it has a different alias there |
| Both requests in 2.14 served by the same model | The rule values or operators do not split at the `max_tokens` values you sent, or one rule points at the wrong Resource |
| `502`, `503`, or a timeout from one model | That model's upstream did not answer. For your Section 1 model, check that the instance is still **Active**. |
| `429` | You hit the profile's request rate limit. Wait for the window to pass, then run the cell again. |


## Wrap-Up

You have now used the LLM Gateway three ways:

| Notebook | What you called | Who chose the model |
|---|---|---|
| `02_Use_the_LLM_Gateway.ipynb` | One alias for one model | You |
| This notebook, 2.11 | Several aliases, one for each model | You, by changing one word |
| This notebook, 2.14 | One Routed Resource alias | The gateway's rules |

**When you are done, stop your Section 1 instance** to release its GPU (Section 1.9).

## Related Docs

- [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [LLM Gateway settings: Resources, Routed Resources, and Profiles](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway)
- [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)